In [1]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import talib
import yfinance as yf
import pandas as pd

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
cl = yf.Ticker('CL=F')
data = cl.history('max')

In [3]:
# Calculate the split point
split_idx = int(len(data) * 0.8)

# Split the data
iss_data = data.iloc[:split_idx]
oss_data = data.iloc[split_idx:]

print(f"ISS Range: {iss_data.index[0]} to {iss_data.index[-1]}")
print(f"OSS Range: {oss_data.index[0]} to {oss_data.index[-1]}")

ISS Range: 2000-08-23 00:00:00-04:00 to 2021-08-06 00:00:00-04:00
OSS Range: 2021-08-09 00:00:00-04:00 to 2026-10-30 00:00:00-04:00


In [4]:
class EMARSI_long(Strategy):

    fast_window = 9
    slow_window = 21
    rsi_window = 14
    filter_window = 200
    
    def init(self):
        self.ema_fast = self.I(talib.EMA, self.data.Close, self.fast_window)
        self.ema_slow = self.I(talib.EMA, self.data.Close, self.slow_window)
        self.rsi = self.I(talib.RSI, self.data.Close, self.rsi_window)

    def next(self):
        price = self.data.Close[-1]
        curr_rsi = self.rsi[-1]
        
        if crossover(self.ema_fast, self.ema_slow) and curr_rsi>50:
            self.buy()

        if crossover(self.ema_slow, self.ema_fast):
            self.position.close()

In [5]:
bt_long = Backtest(iss_data, EMARSI_long, cash=10000)

In [6]:
stats_long = bt_long.run()

Backtest.run:   0%|          | 0/5239 [00:00<?, ?bar/s]

In [7]:
stats_long

Start                     2000-08-23 00:00...
End                       2021-08-06 00:00...
Duration                   7653 days 00:00:00
Exposure Time [%]                    59.77186
Equity Final [$]                  58648.35634
Equity Peak [$]                   69171.68105
Return [%]                          486.48356
Buy & Hold Return [%]               101.11928
Return (Ann.) [%]                     8.84442
Volatility (Ann.) [%]                27.28657
CAGR [%]                              5.99792
Sharpe Ratio                          0.32413
Sortino Ratio                          0.5176
Calmar Ratio                          0.22663
Alpha [%]                                 NaN
Beta                                      NaN
Max. Drawdown [%]                   -39.02499
Avg. Drawdown [%]                    -6.81677
Max. Drawdown Duration     2476 days 00:00:00
Avg. Drawdown Duration      103 days 00:00:00
# Trades                                  108
Win Rate [%]                      

In [8]:
bt_oss_long = Backtest(oss_data, EMARSI_long, cash=10000)

In [9]:
stats_oss_long = bt_oss_long.run()

Backtest.run:   0%|          | 0/1295 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Temp\ipykernel_35996\1986551154.py:1: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats_oss_long = bt_oss_long.run()


In [10]:
stats_oss_long

Start                     2021-08-09 00:00...
End                       2026-10-30 00:00...
Duration                   1908 days 00:00:00
Exposure Time [%]                    44.52888
Equity Final [$]                  10026.74943
Equity Peak [$]                   16243.45948
Return [%]                            0.26749
Buy & Hold Return [%]                52.33358
Return (Ann.) [%]                     0.05117
Volatility (Ann.) [%]                31.25502
CAGR [%]                              0.03529
Sharpe Ratio                          0.00164
Sortino Ratio                         0.00231
Calmar Ratio                          0.00087
Alpha [%]                           -28.88197
Beta                                  0.55699
Max. Drawdown [%]                   -58.71083
Avg. Drawdown [%]                    -8.73587
Max. Drawdown Duration     1697 days 00:00:00
Avg. Drawdown Duration      194 days 00:00:00
# Trades                                   27
Win Rate [%]                      